# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import logging
import json
from pathlib import Path

# 2. Módulos de Terceiros
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T # ArrayType, StructType, StructField, StringType, BooleanType

# 3. Teletools/Teleutils
from teleutils.core.extractors import CDRParquetExtractor, CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
# Configuração para exibir logs no output da célula
logging.basicConfig(
 level=logging.INFO,
 format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
 datefmt="%H:%M:%S",
)

# Reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
# logging.getLogger("py4j").setLevel(logging.WARNING)
# logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark

In [ ]:
spark = (
 SparkSession
 .builder
 .appName("SessaoLocal")
 .master("local[12]")
 .config("spark.driver.memory", "24g")
 .getOrCreate()
)
spark

# Análise

## Config

In [ ]:
from teleutils.core.orchestrators import load_config
file_configs_example = "/workspaces/teleutils/src/teleutils/core/orchestrators/etl.config.example.json"
configs = load_config(file_configs_example)
configs[0]

In [ ]:
config_algar = [
    (i, config)
    for i, config in enumerate(config_example)
    if "algar" in config["output_file_prefix"]
]
config_algar

In [ ]:
config_oi = [
    (i, config)
    for i, config in enumerate(config_example)
    if "oi" in config["output_file_prefix"]
]
config_oi

## Extração e transformação

In [ ]:
config_local = [
    {
        'source_folder': '/assets/data/cdr/teleparser/Semana110/algar/voz',
        'pattern': '*.parquet',
        'schema': 'smp_nokia_gsm',
        'extracted_folder': '/assets/data/temp/extracted',
        'transformed_folder': '/assets/data/temp/transformed',
        'output_file_prefix': 'algar_smp_nokia_gsm'
    },
    {
        'source_folder': '/assets/monitoring/cdr/cdr_bruto/Semana110/Algar/FIXA/NGN',
        'pattern': '*.gz',
        'schema': 'stfc_huawei_ngn',
        'extracted_folder': '/assets/data/temp/extracted',
        'transformed_folder': '/assets/data/temp/transformed',
        'output_file_prefix': 'algar_stfc_huawei_ngn'
    }
]